## Testing Sequence Counts


Calculate the total number of unique sequences in a given data file.

In [78]:
import os
import shutil
import pandas as pd
import numpy as np

In [79]:
def load_codoncounts(filepath):
    """Load in the dataframe for the codoncounts file"""
    df = pd.read_csv(filepath)
    column_names = df.columns.tolist()
    column_names = column_names[2:]
    wildtypes = df["wildtype"].tolist()
    df = df.drop(columns=["site", "wildtype"])
    df_array = df.to_numpy()
    return df_array, column_names, wildtypes

def count_unique_proteins(filepath=None, codon_array=None, 
                          column_names=None, wildtypes=None):
    """Count the number of unique proteins in the codon array"""
    if filepath is not None:
        codon_array, column_names, wildtypes = load_codoncounts(filepath)
    else:
        assert codon_array is not None
        assert column_names is not None
        assert wildtypes is not None
    
    count_unique = 0
    # iterate through each row
    for i in range(codon_array.shape[0]):
        row = codon_array[i]
        for j in range(row.shape[0]):
            if row[j] > 0 and column_names[j] != wildtypes[i]:
                count_unique += 1
    return count_unique

def get_reference_sequence(filepath):
    """Get the reference sequence from a text file"""
    with open(filepath, 'r') as f:
        reference_sequence = f.read().strip()
    
    reference_protein_sequence = ""
    for i in range(0, len(reference_sequence), 3):
        codon = reference_sequence[i:i+3]
        aa = CODON2AA.get(codon, 'X')  # Use 'X' for unknown codons
        reference_protein_sequence += aa
    return reference_protein_sequence


def get_day_estimate(filepath):
    codon_array, column_names, wildtypes = load_codoncounts(filepath)
    total_unique = count_unique_proteins(codon_array=codon_array, 
                                         column_names=column_names, 
                                         wildtypes=wildtypes)
    row_number = codon_array.shape[0] # number of sites
    day_estimate = total_unique * row_number / 100000
    return day_estimate
    


## Reconstructing the Amino Acid Sequences from the codon counts files

In [80]:
pwd = os.getcwd()

data_dir = pwd + '/data/raw_data/'
test_file = data_dir + 'BF520_mutDNA-1_codoncounts.csv'
reference_sequence_file = data_dir + "BF520_Reference_sequence.txt"

In [81]:
# The ending file has more unique proteins than the beginning? I guess some mutations occured

print(count_unique_proteins(filepath=data_dir+'BF520_mutDNA-1_codoncounts.csv'))
print(count_unique_proteins(filepath=data_dir+'BF520_mutDNA-3_codoncounts.csv'))
print(get_day_estimate(test_file))

38187
38307
252.79794


In [82]:
CODON2AA = {'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',            # Map from codons to amino acids
            'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
            'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
            'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
            'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
            'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
            'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
            'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
            'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
            'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
            'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
            'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
            'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
            'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
            'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
            'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W' }

In [88]:
# create a file to write the protein sequences to.

output_file = pwd + '/data/sequence_data/testBF520_protein_sequences.csv'



In [90]:
def write_codon_begin_and_end(filepath1, filepath2, output_path, ref_pro_seq):
    codon_array_1 , column_names_1, wildtypes_1 = load_codoncounts(filepath1)
    codon_array_2 , column_names_2, wildtypes_2 = load_codoncounts(filepath2)
    assert (column_names_1 == column_names_2), "Column names do not match"
    assert (wildtypes_1 == wildtypes_2), "Wildtypes do not match"
    assert (codon_array_1.shape == codon_array_2.shape), "Shapes do not match"
    
    column_names_aa = [CODON2AA.get(codon, 'X') for codon in column_names_1]
    wildtypes_aa = [CODON2AA.get(codon, 'X') for codon in wildtypes_1]
    
    
    with open (output_path, 'w') as f:
        f.write("StartNumber,EndNumber,ProteinSequence\n")
        # iterate through each row
        for i in range(codon_array_1.shape[0]):
            row1 = codon_array_1[i]
            row2 = codon_array_2[i]
            for j in range(row1.shape[0]):
                # For now, only do those that start nonzero
                if row1[j] != 0 and column_names_aa[j] != wildtypes_aa[i]:
                    num_start = row1[j]
                    num_end = row2[j]
                    amino_acid = column_names_aa[j]
                    new_protein_sequence = ref_pro_seq[:i] + amino_acid + ref_pro_seq[i+1:]
                    f.write(f"{num_start},{num_end},{new_protein_sequence}\n")
    print("Wrote to " + output_file)

In [92]:
write_codon_begin_and_end(data_dir+'BF520_mutDNA-1_codoncounts.csv',
                            data_dir+'BF520_mutDNA-3_codoncounts.csv',
                            output_file,
                            get_reference_sequence(reference_sequence_file))

Wrote to /Users/dylanwells/CodingProjects/popDMS/esmDMS/data/sequence_data/testBF520_protein_sequences.csv
